# Stacks & Queues — Order-Restricted Containers

A **stack** and a **queue** both store a sequence but expose only one or two ends, trading random access for $O(1)$ insertion and removal at those ends. A stack is **LIFO** (last in, first out); a queue is **FIFO** (first in, first out). Every operation below records a snapshot so the container's evolution can be replayed and watched, not merely described.

$$ \text{push/pop/enqueue/dequeue} = O(1), \qquad \text{search} = O(n). $$

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.figsize'] = (9, 4.5)

# Operations record snapshot frames; a Play button + slider replays them.
def make_player(n_steps, render, label='step'):
    slider = widgets.IntSlider(value=0, min=0, max=n_steps-1, description=label,
                               continuous_update=False, layout=widgets.Layout(width='60%'))
    play = widgets.Play(value=0, min=0, max=n_steps-1, interval=550)
    widgets.jslink((play, 'value'), (slider, 'value'))
    out = widgets.interactive_output(render, {'k': slider})
    display(widgets.HBox([play, slider]), out)

## Stack — Push and Pop at One End (LIFO)

Both operations act on the **top** of the stack: `push` adds above the current top, `pop` removes it. Because nothing below the top ever moves, each operation is $O(1)$. Stepping through a mixed program shows the top pointer rising and falling and the LIFO discipline — the value popped is always the most recently pushed.

$$ \text{top} \leftarrow \text{top} \pm 1, \qquad \text{pop returns } a[\text{top}]. $$

In [2]:
def run_stack(program):
    stack = []; frames = [(list(stack), None, None, 'start')]
    for op, val in program:
        if op == 'push':
            stack.append(val)
            frames.append((list(stack), len(stack)-1, 'push', f'push {val}'))
        else:
            removed = stack.pop() if stack else None
            note = f'pop -> {removed}' if removed is not None else 'pop (empty)'
            frames.append((list(stack), len(stack), 'pop', note))
    return frames

program = [('push',3),('push',7),('push',1),('pop',None),('push',9),('pop',None),('pop',None),('push',5)]
frames_stk = run_stack(program)

def draw_stack(k):
    stack, hot, kind, note = frames_stk[k]
    fig, ax = plt.subplots(figsize=(4.5, 5))
    for i, v in enumerate(stack):
        c = 'tomato' if i == hot and kind == 'push' else 'lightsteelblue'
        ax.add_patch(plt.Rectangle((0, i), 1, 0.9, facecolor=c, edgecolor='k'))
        ax.text(0.5, i+0.45, str(v), ha='center', va='center', fontsize=11)
    if stack:
        ax.annotate('top', xy=(1.05, len(stack)-1+0.45), va='center', color='tomato', fontsize=10)
    ax.set_xlim(-0.2, 1.8); ax.set_ylim(-0.2, 9)
    ax.set_title(f'step {k}/{len(frames_stk)-1}: {note}'); ax.axis('off'); plt.show()

make_player(len(frames_stk), draw_stack)

Output()

## Stack in Action — Matching Brackets

A classic use: scanning a string and pushing each opening bracket, popping when a matching closer appears. The expression is balanced iff every closer finds its partner and the stack ends empty. Step through to watch the stack mirror the nesting depth.

$$ \text{balanced} \iff \forall \text{ prefixes: pops} \le \text{pushes} \ \wedge\ \text{final depth}=0. $$

In [3]:
PAIRS = {')':'(', ']':'[', '}':'{'}

def check_brackets(s):
    stack = []; frames = [(list(stack), 0, None, 'start', None)]
    ok = True
    for idx, ch in enumerate(s):
        status = None
        if ch in '([{':
            stack.append(ch); status = 'push'
        elif ch in ')]}':
            if stack and stack[-1] == PAIRS[ch]:
                stack.pop(); status = 'match'
            else:
                status = 'error'; ok = False
        frames.append((list(stack), idx, status, f"read '{ch}'", ch))
        if not ok: break
    final = ok and not stack
    frames.append((list(stack), len(s), 'done', 'BALANCED' if final else 'NOT balanced', None))
    return frames

expr_s = widgets.Dropdown(options=['(a[b]{c})', '([)]', '{[()]}', '(((', 'a+b)*c'],
                          value='(a[b]{c})', description='expr')
brk_area = widgets.Output()

def relaunch_brk(*_):
    s = expr_s.value; frames = check_brackets(s)
    def draw(k):
        stack, pos, status, note, ch = frames[k]
        fig, (axs, axt) = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={'width_ratios':[2,1]})
        # string with cursor
        for i, c in enumerate(s):
            col = 'tomato' if i == pos else ('lightgray' if i < pos else 'black')
            axs.text(i, 0, c, ha='center', va='center', fontsize=16, color=col, family='monospace')
        axs.set_xlim(-1, len(s)); axs.set_ylim(-1, 1)
        axs.set_title(f'step {k}/{len(frames)-1}: {note}'); axs.axis('off')
        # stack
        for i, v in enumerate(stack):
            col = 'palegreen' if status == 'match' and i == len(stack) else 'lightsteelblue'
            axt.add_patch(plt.Rectangle((0, i), 1, 0.9,
                          facecolor='tomato' if (status=='push' and i==len(stack)-1) else 'lightsteelblue',
                          edgecolor='k'))
            axt.text(0.5, i+0.45, v, ha='center', va='center', fontsize=12)
        axt.set_xlim(-0.3, 1.6); axt.set_ylim(-0.2, 6)
        axt.set_title('stack'); axt.axis('off')
        plt.tight_layout(); plt.show()
    brk_area.clear_output(wait=True)
    with brk_area: make_player(len(frames), draw)

expr_s.observe(relaunch_brk, 'value')
display(expr_s, brk_area)
relaunch_brk()

Dropdown(description='expr', options=('(a[b]{c})', '([)]', '{[()]}', '(((', 'a+b)*c'), value='(a[b]{c})')

Output()

## Queue — Enqueue at the Rear, Dequeue at the Front (FIFO)

A queue adds at the **rear** and removes from the **front**, so elements leave in arrival order. Stepping through shows both ends moving independently and the FIFO discipline: the value dequeued is always the oldest still present.

$$ \text{enqueue: rear} \leftarrow \text{rear}+1, \qquad \text{dequeue returns front, front} \leftarrow \text{front}+1. $$

In [4]:
from collections import deque

def run_queue(program):
    q = deque(); frames = [(list(q), None, None, 'start')]
    for op, val in program:
        if op == 'enq':
            q.append(val)
            frames.append((list(q), 'rear', 'enq', f'enqueue {val}'))
        else:
            removed = q.popleft() if q else None
            note = f'dequeue -> {removed}' if removed is not None else 'dequeue (empty)'
            frames.append((list(q), 'front', 'deq', note))
    return frames

qprogram = [('enq',3),('enq',7),('enq',1),('deq',None),('enq',9),('deq',None),('enq',5),('deq',None)]
frames_q = run_queue(qprogram)

def draw_queue(k):
    q, end, kind, note = frames_q[k]
    fig, ax = plt.subplots(figsize=(9, 2.6))
    for i, v in enumerate(q):
        c = 'lightsteelblue'
        if kind == 'enq' and i == len(q)-1: c = 'tomato'
        ax.add_patch(plt.Rectangle((i, 0), 0.9, 1, facecolor=c, edgecolor='k'))
        ax.text(i+0.45, 0.5, str(v), ha='center', va='center', fontsize=11)
    if q:
        ax.annotate('front', xy=(0.45, 1.15), ha='center', color='seagreen', fontsize=10)
        ax.annotate('rear', xy=(len(q)-1+0.45, 1.15), ha='center', color='tomato', fontsize=10)
    ax.set_xlim(-0.3, 6); ax.set_ylim(-0.3, 1.6)
    ax.set_title(f'step {k}/{len(frames_q)-1}: {note}'); ax.axis('off'); plt.show()

make_player(len(frames_q), draw_queue)

Output()

## Circular Queue — Reusing a Fixed Buffer

A naive array queue wastes space as `front` advances. A **circular** queue wraps indices with modular arithmetic, reusing freed slots in a buffer of capacity $C$. Step through to watch `front` and `rear` chase each other around the ring; the full condition appears when advancing `rear` would collide with `front`.

$$ \text{rear} \leftarrow (\text{rear}+1) \bmod C, \qquad \text{front} \leftarrow (\text{front}+1) \bmod C. $$

In [5]:
def run_circular(program, C):
    buf = [None]*C; front = 0; rear = 0; size = 0
    frames = [(list(buf), front, rear, size, 'start')]
    for op, val in program:
        if op == 'enq':
            if size < C:
                buf[rear] = val; rear = (rear+1) % C; size += 1
                frames.append((list(buf), front, rear, size, f'enqueue {val}'))
            else:
                frames.append((list(buf), front, rear, size, f'enqueue {val} REJECTED (full)'))
        else:
            if size > 0:
                removed = buf[front]; buf[front] = None; front = (front+1) % C; size -= 1
                frames.append((list(buf), front, rear, size, f'dequeue -> {removed}'))
            else:
                frames.append((list(buf), front, rear, size, 'dequeue (empty)'))
    return frames

cprogram = [('enq',1),('enq',2),('enq',3),('deq',None),('deq',None),
            ('enq',4),('enq',5),('enq',6),('enq',7),('deq',None),('enq',8)]
C_s = widgets.IntSlider(value=5, min=4, max=7, description='capacity C')
circ_area = widgets.Output()

def relaunch_circ(*_):
    C = C_s.value; frames = run_circular(cprogram, C)
    angles = np.linspace(np.pi/2, np.pi/2 - 2*np.pi, C, endpoint=False)
    xs, ys = np.cos(angles), np.sin(angles)
    def draw(k):
        buf, front, rear, size, note = frames[k]
        fig, ax = plt.subplots(figsize=(5.5, 5.5))
        for i in range(C):
            filled = buf[i] is not None
            ax.add_patch(plt.Circle((xs[i], ys[i]), 0.22,
                         facecolor='lightsteelblue' if filled else 'whitesmoke', edgecolor='k'))
            ax.text(xs[i], ys[i], '' if buf[i] is None else str(buf[i]),
                    ha='center', va='center', fontsize=11)
            ax.text(xs[i]*1.42, ys[i]*1.42, str(i), ha='center', va='center', fontsize=8, color='gray')
        ax.annotate('F', xy=(xs[front]*0.6, ys[front]*0.6), ha='center', va='center',
                    color='seagreen', fontsize=13, weight='bold')
        ax.annotate('R', xy=(xs[rear % C]*0.78, ys[rear % C]*0.78), ha='center', va='center',
                    color='tomato', fontsize=13, weight='bold')
        ax.set_xlim(-1.7, 1.7); ax.set_ylim(-1.7, 1.7); ax.set_aspect('equal')
        ax.set_title(f'step {k}/{len(frames)-1} . size={size}/{C}\n{note}'); ax.axis('off'); plt.show()
    circ_area.clear_output(wait=True)
    with circ_area: make_player(len(frames), draw)

C_s.observe(relaunch_circ, 'value')
display(C_s, circ_area)
relaunch_circ()

IntSlider(value=5, description='capacity C', max=7, min=4)

Output()

## Building a Queue from Two Stacks

A FIFO queue can be simulated with two LIFO stacks: an **in** stack receives pushes; when a dequeue is needed and **out** is empty, the entire **in** stack is poured into **out**, reversing the order so the oldest element surfaces. Each element is moved at most once between stacks, giving $O(1)$ *amortized* per operation. Step through to watch the transfer happen only when **out** runs dry.

$$ \text{amortized cost} = O(1), \qquad \text{worst single dequeue} = O(n). $$

In [6]:
def run_two_stacks(program):
    sin, sout = [], []
    frames = [(list(sin), list(sout), None, 'start')]
    for op, val in program:
        if op == 'enq':
            sin.append(val)
            frames.append((list(sin), list(sout), 'in', f'enqueue {val} -> in'))
        else:
            if not sout:
                while sin:                       # transfer, reversing order
                    sout.append(sin.pop())
                    frames.append((list(sin), list(sout), 'move', 'pour in -> out'))
            removed = sout.pop() if sout else None
            note = f'dequeue -> {removed}' if removed is not None else 'dequeue (empty)'
            frames.append((list(sin), list(sout), 'out', note))
    return frames

tprogram = [('enq',1),('enq',2),('enq',3),('deq',None),('deq',None),('enq',4),('deq',None),('deq',None)]
frames_two = run_two_stacks(tprogram)

def draw_two(k):
    sin, sout, kind, note = frames_two[k]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 5))
    for ax, data, name, hot in [(ax1, sin, 'in', kind=='in'), (ax2, sout, 'out', kind=='out')]:
        for i, v in enumerate(data):
            top = (i == len(data)-1)
            c = 'tomato' if (top and hot) else 'lightsteelblue'
            ax.add_patch(plt.Rectangle((0, i), 1, 0.9, facecolor=c, edgecolor='k'))
            ax.text(0.5, i+0.45, str(v), ha='center', va='center', fontsize=11)
        ax.set_xlim(-0.3, 1.6); ax.set_ylim(-0.2, 6)
        ax.set_title(name); ax.axis('off')
    fig.suptitle(f'step {k}/{len(frames_two)-1}: {note}')
    plt.tight_layout(); plt.show()

make_player(len(frames_two), draw_two)

Output()